# Logits Alignments

In [1]:
def default_params(): 
    return {
        'current_model': 'M1',
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/extraction',
            'transformation': 'RenameVariable-2',
            'content_column': 'code',
            'sampling_size': 500,
        },
        
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'raw_logits_path' : '/workspaces/CodeSmells/datax/code_smells/logits',
        'alignments_path': '/workspaces/CodeSmells/data/extension/alignments',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
      
        'causal_models': {
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'microsoft/Phi-3.5-mini-instruct', #https://huggingface.co/microsoft/Phi-3.5-mini-instruct 
            'M4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B
            'M5' : 'facebook/incoder-6B', #https://huggingface.co/facebook/incoder-6B
            'M6' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b 
            'M7' : 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B
            'M8' : 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', #https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import random
import numpy as np
from statistics import mean, median
import os
import torch
import gc
from difflib import SequenceMatcher

In [3]:
from datasets import load_dataset, Dataset

In [4]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

2025-02-19 20:37:13.585504: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739997433.604521  466604 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739997433.610612  466604 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-19 20:37:13.630837: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [6]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}/{params['dataset']['transformation']}"
create_folder(log_file)
log_file += '/align_aggr.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [7]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### Model Loading

In [8]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     match params['quantization']:
               case 'int4':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
               case 'int8':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
               case 'float32':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
               case 'float16':
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
               case _: 
                    model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [9]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

#### Load Dataset

In [10]:
df_actual_ntp = pd.read_json(f"{params['raw_logits_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}/raw_logits.json")

In [11]:
df_actual_ntp.head(2)

,id,commit_id,repo,path,file_name,commit_message,url,language,category,code,...,ast_levels,n_ast_nodes,n_ast_errors,n_identifiers,input_ids,input_lenght,max_prob,min_prob,actual_prob,loss
0,256261,a59bca366174d9c692fa19750c24d65f47660ef7,haystack,haystack/modeling/training/base.py,base.py,Apply black formatting (#2115)\n\n* Testing bl...,https://github.com/deepset-ai/haystack.git,Python,Warning,def _get_state_dict(result):\n \n ...,...,9,206,0,19,"[822, 903, 657, 29918, 3859, 29918, 8977, 2989...",291,"[[<PRE>, 0.7492886186000001], [module, 0.47944...","[[<s>, 0.0], [$}, 1e-10], [oreferrer, 0.0], [o...","[[def, 0.0007611339000000001], [_, 0.009010307...",0.737458
1,305801,6f564e4f514b56bce281ec7e82703cfbff87b417,core,homeassistant/components/ring/binary_sensor.py,binary_sensor.py,Improve entity type hints [r] (#77874),https://github.com/home-assistant/core.git,Python,Warning,async def async_added_to_hass(result) -> resul...,...,10,60,0,6,"[7465, 822, 7465, 29918, 23959, 29918, 517, 29...",72,"[[<PRE>, 0.7492926717], [\n, 0.145589932800000...","[[<s>, 0.0], [oreferrer, 4e-10], [$}, 1.100000...","[[async, 1.08118e-05], [def, 0.0006238665], [a...",1.948516


#### Token Binding

In [12]:
def find_range_of_indexes(positions, search_range):
    """
    Finds the range of indexes in the positions array where the search_range is fully included.
    
    Args:
    - positions: A list of tuples, where each tuple is (start_position, end_position) (inclusive).
    - search_range: A tuple (start_position, end_position), where start_position is inclusive and end_position is exclusive.
    
    Returns:
    - A tuple (start_index, end_index) representing the range of indexes in the positions array where the search_range is included.
    """
    start, end = search_range
    start_index = -1
    end_index = -1

    for i, (pos_start, pos_end) in enumerate(positions):
        if pos_start <= start <= pos_end:  # Find the start of the range
            start_index = i
        if pos_start <= end - 1 <= pos_end and pos_end>=end:  # Find the end of the range
            end_index = i
            break

    if start_index != -1 and end_index != -1:
        return (start_index, end_index)
    else:
        return None  # If no range is found

In [13]:
def get_substring_positions(code: str, code_smell: str, start):
    """
    Calculate the start and end positions of the substring based on line and column information.

    Parameters:
    text (str): The input string containing multiple lines.
    start (tuple): A tuple of (start_line, start_column) indicating the start position.
    end (tuple): A tuple of (end_line, end_column) indicating the end position.

    Returns:
    tuple: A tuple containing (start_position, end_position) of the substring in the input string.
    """
    lines = code.split('\n')  # Split the string into lines

    # Calculate the character position for the start of the substring
    start_line, start_column = start
    
    try:
        start_position = sum(len(lines[i]) + 1 for i in range(start_line - 1)) + start_column
    except:
        start_position = code.find(code_smell)

    if start_line > len(lines) or start_position>= len(code): 
        start_position = code.find(code_smell)

    if start_position == -1:
        match = SequenceMatcher(None, code, code_smell).find_longest_match()
        start_position= match.a
        end_position = match.a + match.size
    else:
        end_position = start_position + len(code_smell)
        end_position = len(code) if end_position >= len(code) else end_position
    

    return (start_position, end_position)

In [14]:
def find_code_smell_logits(code, code_smell_pos, logits_array, tokenizer):
    indexes_range = find_range_of_indexes(tokenizer.encode_plus(code, return_offsets_mapping=True, add_special_tokens=False)['offset_mapping'], code_smell_pos)
    return logits_array[indexes_range[0]:indexes_range[1]+1]

In [15]:
df_actual_ntp.columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'commit_message', 'url',
       'language', 'category', 'code', 's_msg_id', 's_line', 's_column',
       's_end_line', 's_end_column', 's_code', 'n_whitespaces', 'n_words',
       'vocab_size', 'fun_name', 'complexity', 'nloc', 'token_counts',
       'ast_errors', 'ast_levels', 'n_ast_nodes', 'n_ast_errors',
       'n_identifiers', 'input_ids', 'input_lenght', 'max_prob', 'min_prob',
       'actual_prob', 'loss'],
      dtype='object')

In [16]:
df_actual_ntp['code_smell_pos'] = df_actual_ntp.apply(lambda row: get_substring_positions(row['code'], row['s_code'], (row['s_line'], row['s_column'])), axis=1)

In [17]:
df_actual_ntp.columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'commit_message', 'url',
       'language', 'category', 'code', 's_msg_id', 's_line', 's_column',
       's_end_line', 's_end_column', 's_code', 'n_whitespaces', 'n_words',
       'vocab_size', 'fun_name', 'complexity', 'nloc', 'token_counts',
       'ast_errors', 'ast_levels', 'n_ast_nodes', 'n_ast_errors',
       'n_identifiers', 'input_ids', 'input_lenght', 'max_prob', 'min_prob',
       'actual_prob', 'loss', 'code_smell_pos'],
      dtype='object')

In [18]:
# Alignments
df_actual_ntp['code_smell_actual_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['actual_prob'], tokenizer), axis=1)
df_actual_ntp['code_smell_max_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['max_prob'], tokenizer), axis=1)
df_actual_ntp['code_smell_min_logits'] = df_actual_ntp.apply(lambda row: find_code_smell_logits(row['code'], row['code_smell_pos'], row['min_prob'], tokenizer), axis=1)

In [19]:
## Aggregations - median
df_actual_ntp['code_smell_actual_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), axis=1)
df_actual_ntp['code_smell_max_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']]), axis=1)
df_actual_ntp['code_smell_min_prob_median'] = df_actual_ntp.apply(lambda row: median([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), axis=1)

In [20]:
## Aggregations  - mean
df_actual_ntp['code_smell_actual_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_actual_logits']]), axis=1)
df_actual_ntp['code_smell_max_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_max_logits']]), axis=1)
df_actual_ntp['code_smell_min_prob_mean'] = df_actual_ntp.apply(lambda row: mean([logit_tuple[1] for logit_tuple in row['code_smell_min_logits']]), axis=1)

In [21]:
df_actual_ntp

,id,commit_id,repo,path,file_name,commit_message,url,language,category,code,...,code_smell_pos,code_smell_actual_logits,code_smell_max_logits,code_smell_min_logits,code_smell_actual_prob_median,code_smell_max_prob_median,code_smell_min_prob_median,code_smell_actual_prob_mean,code_smell_max_prob_mean,code_smell_min_prob_mean
0,256261,a59bca366174d9c692fa19750c24d65f47660ef7,haystack,haystack/modeling/training/base.py,base.py,Apply black formatting (#2115)\n\n* Testing bl...,https://github.com/deepset-ai/haystack.git,Python,Warning,def _get_state_dict(result):\n \n ...,...,"(29, 47)","[[ , 0.0001560742], [\n, 0.8587702513000...","[[ , 0.9265300035], [\n, 0.8587702513000001],...","[[oreferrer, 0.0], [ال, 0.0], [oreferrer, 0.0]...",0.293775,0.702377,0.0,0.361619,0.631538,0.000000e+00
1,305801,6f564e4f514b56bce281ec7e82703cfbff87b417,core,homeassistant/components/ring/binary_sensor.py,binary_sensor.py,Improve entity type hints [r] (#77874),https://github.com/home-assistant/core.git,Python,Warning,async def async_added_to_hass(result) -> resul...,...,"(102, 141)","[[ , 0.36418789630000004], [result, 0.020...","[[ , 0.36418789630000004], [self, 0.27008...","[[ightarrow, 0.0], [java, 0.0], [oreferrer, 0....",0.032690,0.364188,0.0,0.185893,0.387583,0.000000e+00
2,70649,5fe901e5d86ed02dbbb63039a897582951266afd,wagtail,wagtail/admin/tests/pages/test_edit_page.py,test_edit_page.py,Fix commenting thread notifications being sent...,https://github.com/wagtail/wagtail.git,Python,Convention,def test_new_comment(result):\n result ...,...,"(1308, 1437)","[[ , 0.5153093934], [result, 0.9591836929...","[[ , 0.5153093934], [result, 0.9591836929...","[[oreferrer, 0.0], [throw, 0.0], [ITableView, ...",0.959184,0.959184,0.0,0.783428,0.852065,3.321893e-17
3,151753,bdfedb5fcb02b88c600ef25c88bbb5d939b8bd0a,freqtrade,freqtrade/freqai/RL/BaseReinforcementLearningM...,BaseReinforcementLearningModel.py,Improve typehints / reduce warnings from mypy,https://github.com/freqtrade/freqtrade.git,Python,Warning,"def __init__(result, **result) -> result:\n ...",...,"(766, 806)","[[ , 0.902915597], [if, 0.0678886771], [r...","[[ , 0.902915597], [result, 0.85789340730...","[[oreferrer, 0.0], [decl, 0.0], [<MID, 0.0], [...",0.767980,0.902916,0.0,0.543116,0.817715,0.000000e+00
4,3868,2282a4ae0221b1fb88e16eca8bc14a166998d2d2,airbyte,airbyte-integrations/connectors/source-hubspot...,streams.py,🎉 Source Hubspot: Migrate to CDK (#10177)\n\n*...,https://github.com/airbytehq/airbyte.git,Python,Warning,"def state(result, result):\n result = r...",...,"(250, 317)","[[ , 0.9683276415000001], [), 0.980689227...","[[ , 0.9683276415000001], [), 0.980689227...","[[oreferrer, 0.0], [};, 2.5844593190000003e-17...",0.537555,0.611231,0.0,0.563394,0.681026,1.360242e-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17273,43471,09f38ad3f6872bae5059a1de226362eb358c4a7a,airflow,tests/providers/microsoft/azure/operators/test...,test_asb.py,Implement Azure Service Bus Queue Operators (#...,https://github.com/apache/airflow.git,Python,Convention,"def test_send_message_queue(result, result):\n...",...,"(294, 395)","[[result, 0.2436789423], [=, 0.4119559526], [[...","[[assert, 0.37523525950000003], [., 0.51229745...","[[align, 0.0], [ITableView, 0.0], [<MID, 0.0],...",0.552739,0.701317,0.0,0.530922,0.645097,0.000000e+00
17274,196984,4a6d5d342e1d0111130d4b31708535b862bfacd0,sympy,sympy/printing/repr.py,repr.py,Update the deprecation for Permutation.print_c...,https://github.com/sympy/sympy.git,Python,Convention,"def _print_Permutation(result, result):\n ...",...,"(704, 745)","[[singleton, 0.0733625889], [and, 0.0767336860...","[[singleton, 0.0733625889], [\n, 0.2097381204]...","[[<MID, 0.0], [filters, 0.0], [animation, 0.0]...",0.196023,0.535473,0.0,0.385941,0.522715,0.000000e+00
17275,105,0b8a53bd313abdf484a9d1e3fbd6aad13c0ec857,PySyft,packages/syft/tests/syft/core/tensor/passthrou...,passthrough_test.py,adding tests,https://github.com/OpenMined/PySyft.gi

#### SAVE

In [22]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [23]:
alignments_dir = f"{params['alignments_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}"
create_folder(alignments_dir)
df_actual_ntp.to_json(f"{alignments_dir}/aligned_smells.json")

In [24]:
torch.cuda.empty_cache()
gc.collect()

0